In [ ]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [21]:
summary_generator = AssistantAgent(
    name="Summary_Generator",
    model_client=model_client,
    description="A summary generator",
    system_message="You generate a summary in under 25 words for the given query."
)

summary_reviewer = AssistantAgent(
    name="Summary_Reviewer",
    model_client=model_client,
    description="A summary reviewer",
    system_message="""Generate 1-2 relevant examples based on the summary generated by summary_generator. if you think the summary is fine, just sat 'APPROVED'"""    
)

summary_editor = AssistantAgent(
    name="Summary_Editor",
    model_client=model_client,
    description="A summary editor",
    system_message = """You are a summary editor. You update the summary based on the examples given by summary_reviewer."""
)

In [22]:
my_termination = TextMentionTermination(text='APPROVED')  | MaxMessageTermination(max_messages=10)

team = RoundRobinGroupChat(
    participants=[summary_generator, summary_reviewer, summary_editor],
    max_turns=10,
    termination_condition=my_termination   
)

In [ ]:
async def run_team():
    task = TextMessage(content='Machine Learning',source='user')
    stream = await team.run(task=task)

    for each_agent_message in stream.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")


In [24]:
await run_team()

user : Machine Learning
Summary_Generator : Machine Learning is a field of AI focusing on algorithms that enable computers to learn from and make predictions based on data.
Summary_Reviewer : Example 1: A machine learning model is trained using a dataset of labeled emails to recognize and categorize future incoming emails as either 'spam' or 'not spam.'

Example 2: An e-commerce platform uses machine learning algorithms to analyze users' purchase history and browsing behavior to recommend personalized product suggestions.
Summary_Editor : Machine Learning is a field of AI focused on developing algorithms that enable computers to learn from data and make predictions or decisions. Examples include using models to categorize emails as spam or not and personalizing product recommendations based on user behavior.
Summary_Generator : Machine Learning involves AI algorithms that learn from data to make decisions, like email categorization or personalized product recommendations.
Summary_Revie